# Week 7, Day 5: Checking functions with assert

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w007d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def area(w, h):
    return w * h

print(area(3, 4) == 12)
print(area(0, 5) == 0)
print(area(2.5, 2) == 5.0)

In [ ]:
x = 5
assert x > 0
print("x is positive, carry on")

In [ ]:
x = -3
assert x > 0, f"x should be positive, but it's {x}"
print("never reached")

> **Key idea.** `assert condition, message` does nothing when the condition is true, and stops the program with an `AssertionError` (and your message) when it's false.

> **Brackets that make it always pass.** `assert(x > 0, "message")` looks like a function call, but `assert` isn't a function. Python reads the brackets as a **pair** of values, and a non-empty pair is always truthy, so this assert can **never** fail. Python even warns you about it. Write `assert x > 0, "message"`, with no brackets.

In [ ]:
def area(width, height):
    """Return the area of a rectangle."""
    return width * height

def test_area():
    assert area(3, 4) == 12
    assert area(1, 1) == 1
    assert area(0, 5) == 0, "a zero side gives zero area"
    assert area(2.5, 2) == 5.0

test_area()
print("All tests passed")

> **Tip.** Name test functions `test_` followed by the name of the function they test. Tools like **pytest**, which you'll use in Week 18, find and run every function whose name starts with `test_` automatically.

In [ ]:
def grade(mark):
    """Return A (70+), B (60-69), C (50-59) or F."""
    if mark >= 70:
        return "A"
    if mark >= 60:
        return "B"
    if mark >= 50:
        return "C"
    return "F"

def test_grade():
    assert grade(85) == "A"
    assert grade(65) == "B"
    assert grade(55) == "C"
    assert grade(20) == "F"
    for mark, want in [(70, "A"), (69, "B"), (60, "B"), (59, "C"), (50, "C"), (49, "F")]:
        assert grade(mark) == want, f"grade({mark}) should be {want}, got {grade(mark)}"
    assert grade(100) == "A" and grade(0) == "F"

test_grade()
print("grade passes")

In [ ]:
def initials(name):
    """Return the initials of a name, capitals with full stops."""
    result = ""
    for part in name.split():
        result += part[0].upper() + "."
    return result

def test_initials():
    assert initials("ada lovelace") == "A.L."
    assert initials("  Grace   Brewster Hopper ") == "G.B.H."
    assert initials("Plato") == "P."
    assert initials("") == ""

test_initials()
print("initials passes")

In [ ]:
assert 0.1 + 0.2 == 0.3

In [ ]:
import math
assert abs((0.1 + 0.2) - 0.3) < 1e-9
assert math.isclose(0.1 + 0.2, 0.3)
print("close enough")

In [ ]:
def digit_sum(n):
    """Return the sum of the digits of n. n must be at least 0."""
    assert n >= 0, f"digit_sum needs n >= 0, got {n}"
    total = 0
    while n > 0:
        total += n % 10
        n //= 10
    return total

print(digit_sum(4096))

### Exercise: Your first test

Write `test_double()` with at least four asserts checking `double`: a positive number, zero, a negative number and a float. Then call it and print `ok`.

In [ ]:
def double(n):
    return n * 2

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_written():\n    \"the test exists, has four asserts, and passes\"\n    assert callable(test_double), \"Define test_double().\"\n    assert source.count(\"assert\") >= 4, \"Write at least four asserts.\"\n    expect_output(\"ok\")\ndef test_catches_bugs():\n    \"your test catches a broken double\"\n    g = test_double.__globals__\n    good = g[\"double\"]\n    g[\"double\"] = lambda n: n + n if n > 0 else n\n    try:\n        try:\n            test_double()\n        except AssertionError:\n            return\n        assert False, \"Your test passed a broken double (one that's wrong for zero or negatives). Check those cases.\"\n    finally:\n        g[\"double\"] = good")

### Exercise: Make the tests pass

The tests for `vowel_count` are written. Write the function so that `test_vowel_count()` passes.

In [ ]:
def vowel_count(text):
    """Return the number of vowels (a, e, i, o, u, either case) in text."""
    return 0

def test_vowel_count():
    assert vowel_count("banana") == 3
    assert vowel_count("RHYTHM") == 0
    assert vowel_count("") == 0
    assert vowel_count("AEIOU aeiou") == 10
    assert vowel_count("Python Atlas") == 3

test_vowel_count()
print("All tests passed")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the tests pass\"\n    expect_output(\"All tests passed\")")

### Exercise: Test the boundaries

`ticket_price(age)` returns 0 for under 5, 500 for 5 to 17, 1200 for 18 to 64 and 800 for 65 or over. Write `test_ticket_price()` that checks **every boundary**: the ages just below, on and above each threshold. Your test must pass the correct function and catch each of the broken versions the checks will try.

In [ ]:
def ticket_price(age):
    if age < 5:
        return 0
    if age < 18:
        return 500
    if age < 65:
        return 1200
    return 800

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_passes():\n    \"your test passes the correct function\"\n    test_ticket_price()\ndef test_catches():\n    \"your test catches each broken version\"\n    g = test_ticket_price.__globals__\n    good = g[\"ticket_price\"]\n    broken = [\n        lambda a: 0 if a <= 5 else 500 if a < 18 else 1200 if a < 65 else 800,\n        lambda a: 0 if a < 5 else 500 if a <= 18 else 1200 if a < 65 else 800,\n        lambda a: 0 if a < 5 else 500 if a < 18 else 1200 if a <= 65 else 800,\n        lambda a: 0 if a < 4 else 500 if a < 18 else 1200 if a < 65 else 800,\n    ]\n    try:\n        for k, bad in enumerate(broken, 1):\n            g[\"ticket_price\"] = bad\n            try:\n                test_ticket_price()\n            except AssertionError:\n                continue\n            assert False, f\"Your test passed broken version {k}. Check the ages just below, on and above each boundary.\"\n    finally:\n        g[\"ticket_price\"] = good")

### Exercise: Test first

Do it the test-first way. First write `test_is_valid_pin()` with at least five asserts, for the rule: a PIN is valid if it's exactly 4 characters, all digits, and not all the same digit (so `"1111"` is invalid). Then write `is_valid_pin(pin)` so your test passes.

In [ ]:
def is_valid_pin(pin):
    """Return True if pin is 4 digits, not all the same."""
    return False

def test_is_valid_pin():
    pass

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_function():\n    \"is_valid_pin follows the rule\"\n    for pin, want in [(\"1234\", True), (\"0000\", False), (\"123\", False), (\"12345\", False), (\"12a4\", False), (\"\", False), (\"9899\", True), (\"1112\", True)]:\n        assert is_valid_pin(pin) == want, f\"is_valid_pin({pin!r}) should be {want}.\"\ndef test_your_test():\n    \"your test has at least five asserts and passes\"\n    start = source.find(\"def test_is_valid_pin\")\n    body = source[start:].split(\"\\ndef \")[0]\n    assert body.count(\"assert\") >= 5, \"Write at least five asserts in test_is_valid_pin.\"\n    test_is_valid_pin()")

### Exercise: Close enough

`average` returns floats. Fix the test so that it compares safely, using `math.isclose`, and passes.

In [ ]:
import math

def average(a, b, c):
    return (a + b + c) / 3

def test_average():
    assert average(0.1, 0.2, 0.3) == 0.2
    assert average(1, 2, 3) == 2

test_average()
print("passed")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the test passes\"\n    expect_output(\"passed\")\ndef test_isclose():\n    \"uses isclose\"\n    assert \"isclose\" in source")

### Exercise: A precondition

Add an `assert` as the first line of `percentage` that stops it with the message `total must be above 0` when `total` isn't above 0.

In [ ]:
def percentage(part, total):
    """Return part as a percentage of total. total must be above 0."""
    return part / total * 100

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_normal():\n    \"works normally\"\n    assert percentage(1, 4) == 25.0\ndef test_assert():\n    \"refuses a bad total\"\n    for bad in [0, -5]:\n        try:\n            percentage(1, bad)\n        except AssertionError as e:\n            assert \"total must be above 0\" in str(e), \"Use the message 'total must be above 0'.\"\n        else:\n            assert False, f\"percentage(1, {bad}) should stop with an AssertionError.\"")

### Bug hunt: A test that can't fail

This test passes even though `triple` is wrong. Fix the **test** so that it fails for the broken `triple`; then fix `triple` so that it passes.

In [ ]:
def triple(n):
    return n + 3

def test_triple():
    assert(triple(2) == 6, "triple(2) should be 6")
    assert(triple(0) == 0, "triple(0) should be 0")

test_triple()
print("passed")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the fixed test passes\"\n    expect_output(\"passed\")\ndef test_real_asserts():\n    \"the asserts can fail now\"\n    assert \"assert(\" not in source.replace(\" \", \"\"), \"Remove the brackets: assert is not a function.\"\n    assert triple(5) == 15")

### Bug hunt: Read the failure

`test_reverse_words` fails. Read the `AssertionError` message to see which case is wrong, then fix the **function** (the test is right).

In [ ]:
def reverse_words(s):
    words = s.split(" ")
    out = ""
    for w in words:
        out = w + " " + out
    return out.strip()

def test_reverse_words():
    assert reverse_words("a b c") == "c b a"
    for s, want in [("one", "one"), ("  hello   world ", "world hello")]:
        assert reverse_words(s) == want, f"reverse_words({s!r}) gave {reverse_words(s)!r}, wanted {want!r}"

test_reverse_words()
print("passed")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the test passes\"\n    expect_output(\"passed\")")

### Puzzle: A test function

Arrange the lines into a test for `is_even`, then run it. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Which assert fails?

For each block, write the number of the **first** assert that fails (1, 2 or 3), or `0` if none fails.

```python
# Block A
s = "Atlas"
assert len(s) == 5          # 1
assert s[0].islower()       # 2
assert s.endswith("s")      # 3

# Block B
n = 10
assert n % 2 == 0           # 1
assert 0 < n <= 10          # 2
assert n // 3 == 3          # 3

# Block C
assert ("x", "y")           # 1
assert 0.1 * 3 == 0.3       # 2
assert "" or "z"            # 3
```

In [ ]:
block_a = ___
block_b = ___
block_c = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_blocks():\n    \"each block\"\n    assert block_a == 2, \"'A' is upper case.\"\n    assert block_b == 0, \"All three are true: 10 // 3 is 3.\"\n    assert block_c == 2, \"A non-empty tuple is truthy, so 1 passes; 0.1 * 3 isn't exactly 0.3.\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a test suite that catches bugs

`is_valid_time(text)` should return `True` for a 24-hour time in the form `HH:MM`: exactly 5 characters, two digits, a colon, two digits, hours 00 to 23 and minutes 00 to 59.

Write `test_is_valid_time()`. It must **pass** the correct function given below, and **fail** (with an `AssertionError`) for every one of six broken versions the checks will swap in, each with a different subtle bug. Think hard about boundaries and odd inputs: that's what good testers do.

In [ ]:
def is_valid_time(text):
    """Return True if text is a valid 24-hour time HH:MM."""
    return (len(text) == 5 and text[2] == ":" and text[:2].isdigit() and text[3:].isdigit()
            and int(text[:2]) <= 23 and int(text[3:]) <= 59)

def test_is_valid_time():
    pass

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_passes():\n    \"your test passes the correct function\"\n    test_is_valid_time()\ndef test_catches():\n    \"your test catches every broken version\"\n    def ok(t):\n        return len(t) == 5 and t[2] == \":\" and t[:2].isdigit() and t[3:].isdigit() and int(t[:2]) <= 23 and int(t[3:]) <= 59\n    broken = {\n        \"hours up to 24 allowed\": lambda t: len(t) == 5 and t[2] == \":\" and t[:2].isdigit() and t[3:].isdigit() and int(t[:2]) <= 24 and int(t[3:]) <= 59,\n        \"minutes up to 60 allowed\": lambda t: len(t) == 5 and t[2] == \":\" and t[:2].isdigit() and t[3:].isdigit() and int(t[:2]) <= 23 and int(t[3:]) <= 60,\n        \"hour 23 refused\": lambda t: len(t) == 5 and t[2] == \":\" and t[:2].isdigit() and t[3:].isdigit() and int(t[:2]) < 23 and int(t[3:]) <= 59,\n        \"any separator allowed\": lambda t: len(t) == 5 and t[:2].isdigit() and t[3:].isdigit() and int(t[:2]) <= 23 and int(t[3:]) <= 59,\n        \"one-digit hours allowed\": lambda t: 4 <= len(t) <= 5 and \":\" in t and t.split(\":\")[0].isdigit() and t.split(\":\")[1].isdigit() and int(t.split(\":\")[0]) <= 23 and int(t.split(\":\")[1]) <= 59,\n        \"minute 00 refused\": lambda t: ok(t) and t[3:] != \"00\",\n    }\n    g = test_is_valid_time.__globals__\n    good = g[\"is_valid_time\"]\n    try:\n        for name, bad in broken.items():\n            g[\"is_valid_time\"] = bad\n            try:\n                test_is_valid_time()\n            except AssertionError:\n                continue\n            except Exception:\n                continue\n            assert False, f\"Your test passed a broken version ({name}). Add a case that catches it.\"\n    finally:\n        g[\"is_valid_time\"] = good")

## Warm-up

Three `True`s. You checked `area` by comparing results with what you expected. Today you'll make Python do that checking for you, and shout when something's wrong.

## assert

`assert` states something that **must** be true at that point in the program:

If the condition is true, nothing happens and the program carries on. If it's false, Python stops with an `AssertionError`:

The part after the comma is the **message**, shown with the error. It's optional, but always worth writing: a good message says what was expected and what actually happened.

## Writing tests

A **test** is a function that calls the function being tested with some inputs and asserts that each result is right:

When every assert passes, the test is silent and the last line confirms it. Now break `area` (change `*` to `+`) and run it again: the first failing assert stops everything and tells you which case broke.

### Which cases should a test check?

You learned this in Week 2, Day 5, for decisions. The same thinking applies:

1. **Typical cases**: ordinary inputs where you know the answer.
2. **Boundaries**: the edges, like 0, 1, an empty string, exactly on a threshold.
3. **Special cases**: anything the docstring mentions, or anything unusual: negative numbers, capitals, spaces.

The loop checks every boundary, and its message names the exact case that failed. (The square brackets make a **list** of pairs to loop over; lists are next week.)

### A test is a promise kept

Once a test passes, keep it. Every time you change the function later, run the test again. If you "improve" something and break an old case, the test catches it straight away. This is called **regression testing**, and it's the main reason professionals write tests: not to find today's bugs, but to stop tomorrow's changes from bringing old bugs back.

## Tests first

Here's a surprisingly powerful habit: write the test **before** the function.

1. Write a docstring for the function, and a stub.
2. Write a test full of examples of what it should do. Run it: it fails, because the stub does nothing.
3. Write the body until the test passes.
4. Tidy the code, running the test after each change.

This is called **test-driven development**. Writing the tests first forces you to decide exactly what the function should do before you're distracted by how. And when the test passes, you're done: no need to wonder.

## Comparing floats in tests

You know from Week 1 that floats are rarely exact, so `==` is dangerous in tests too:

Check that the result is close enough instead:

## When to use assert, and when not

`assert` is for checking **your own** reasoning: in tests, and inside functions to state things that should be impossible if the code is right.

If someone calls `digit_sum(-5)`, the assert stops them with a clear message instead of a silently wrong answer.

But **don't** use `assert` to check what a **user** typed. Python can be run with asserts switched off (with the `-O` option), and then they vanish. Checking input is part of your program's normal job: use `if` and a friendly message, as you have been. In Week 11 you'll learn `raise`, the proper way for a function to refuse bad arguments.

| Use `assert` for | Use `if` (or later, `raise`) for |
|---|---|
| tests | checking user input |
| "this can't happen if my code is right" | anything that can go wrong in normal use |

## Practice

## Quiz

## Daily challenge

## Before you go

Take any function you wrote this week and write a test for it with at least five asserts, including the boundaries. Then break the function on purpose and check that your test notices. Tomorrow is the Forge: a project designed top-down and tested function by function, and the weekly quiz.